In [1]:
import pandas as pd

In [8]:
df1 = pd.read_excel("Hospital_Dataset.xlsx")
df2 = pd.read_excel("Patient_Dataset.xlsx")

print(df1.head())
print(df2.head())

  Hospital ID       Hospital Name Hospital Type       City        State  \
0   HOSP00016        C K Hospital       Private     Mumbai  Maharashtra   
1   HOSP00019  Cure Well Hospital       Private  Bengaluru    Karnataka   
2   HOSP00004    Yashoda Hospital       Private  Hyderabad    Telangana   
3   HOSP00017     Dharma Hospital       Private      Delhi        Delhi   
4   HOSP00017     Dharma Hospital       Private      Delhi        Delhi   

         Department Department ID            Doctor  Nurses  Staff Count  ...  \
0  General Medicine       DEPT004  Dr. Arjun Sharma      12          257  ...   
1   General Surgery       DEPT010     Dr. Sneha Rao      15          288  ...   
2       Pulmonology       DEPT005   Dr. Priya Reddy      20          181  ...   
3       Pulmonology       DEPT005   Dr. Rahul Verma      23          170  ...   
4               ICU       DEPT009   Dr. Priya Reddy      34          116  ...   

  Beds Available ICU Beds Bed Number  Bed Occupied         Equ

In [9]:
print(df1.columns)
print(df2.columns)

Index(['Hospital ID', 'Hospital Name', 'Hospital Type', 'City', 'State',
       'Department', 'Department ID', 'Doctor', 'Nurses', 'Staff Count',
       'Patient ID', 'Patient Name', 'Gender', 'Age', 'Admission Date',
       'Discharge Date', 'Diagnosis', 'Treatment', 'Medication',
       'Admission Type', 'Test Result', 'Blood Type', 'Beds Available',
       'ICU Beds', 'Bed Number', 'Bed Occupied', 'Equipment', 'Length of Stay',
       'Room No', 'Billing Amount', 'Insurance Provider', 'Readmission'],
      dtype='str')
Index(['Equipment Usage Duration', 'Patient Analysis', 'Patient Movement',
       'Patient Transfer to Another Department', 'Current Department Duration',
       'Doctor In Charge'],
      dtype='str')


In [12]:
same_columns = df1.columns.intersection(df2.columns)
print(same_columns)

Index([], dtype='str')


In [14]:
merge_files = pd.concat([df1, df2], axis=1)

print(merge_files.head())

  Hospital ID       Hospital Name Hospital Type       City        State  \
0   HOSP00016        C K Hospital       Private     Mumbai  Maharashtra   
1   HOSP00019  Cure Well Hospital       Private  Bengaluru    Karnataka   
2   HOSP00004    Yashoda Hospital       Private  Hyderabad    Telangana   
3   HOSP00017     Dharma Hospital       Private      Delhi        Delhi   
4   HOSP00017     Dharma Hospital       Private      Delhi        Delhi   

         Department Department ID            Doctor  Nurses  Staff Count  ...  \
0  General Medicine       DEPT004  Dr. Arjun Sharma      12          257  ...   
1   General Surgery       DEPT010     Dr. Sneha Rao      15          288  ...   
2       Pulmonology       DEPT005   Dr. Priya Reddy      20          181  ...   
3       Pulmonology       DEPT005   Dr. Rahul Verma      23          170  ...   
4               ICU       DEPT009   Dr. Priya Reddy      34          116  ...   

  Room No Billing Amount Insurance Provider  Readmission  \
0 

In [15]:
merge_files.to_csv("Final_Hospital_Dataset.csv", index=False)
print("Successfully Saved!")

Successfully Saved!


## Cleaning Notebook
This notebook creates a reproducible data-cleaning script `clean_hospital_data.py`, runs the cleaning pipeline on `hospital_cleaned.csv`, and writes `hospital_cleaned_v2_from_notebook.csv`.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
print('pandas', pd.__version__)

In [ ]:
def clean_hospital_data(in_path='hospital_cleaned.csv', out_path='hospital_cleaned_v2_from_notebook.csv'):
    import pandas as pd, numpy as np
    df = pd.read_csv(in_path)
    # drop unnamed columns
    df = df.loc[:, ~df.columns.str.contains('^Unnamed', na=False)]
    # strip whitespace from object/string columns
    for c in df.select_dtypes(include=['object']).columns:
        df[c] = df[c].where(df[c].isna(), df[c].str.strip())
    # standardize casing for selected text columns
    for c in ['Hospital Name','City','State','Department','Doctor','Patient Name','Admission Type','Insurance Provider']:
        if c in df.columns:
            df[c] = df[c].astype(str).str.title().replace({'Nan': np.nan})
    # uppercase ID-like columns
    for c in df.columns:
        if c.lower().endswith('id') or ' id' in c.lower() or c.lower().startswith('hospital id'):
            df[c] = df[c].astype(str).str.upper()
    # parse dates
    for c in ['Admission Date','Discharge Date']:
        if c in df.columns:
            df[c] = pd.to_datetime(df[c], errors='coerce')
    # parse times to HH:MM strings
    for c in ['Admission Time','Discharge Time']:
        if c in df.columns:
            df[c] = pd.to_datetime(df[c], errors='coerce').dt.strftime('%H:%M').fillna('')
    # numeric conversions (best-effort)
    for c in ['Age','Beds Available','ICU Beds','Staff Count','Billing Amount','Length of Stay']:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors='coerce')
    # fill missing insurance with 'Self-Pay'
    if 'Insurance Provider' in df.columns:
        df['Insurance Provider'] = df['Insurance Provider'].replace({'': np.nan}).fillna('Self-Pay')
    # fix discharge before admission by setting discharge = admission when earlier
    if 'Admission Date' in df.columns and 'Discharge Date' in df.columns:
        mask = (df['Discharge Date'].notna()) & (df['Admission Date'].notna()) & (df['Discharge Date'] < df['Admission Date'])
        df.loc[mask, 'Discharge Date'] = df.loc[mask, 'Admission Date']
    # deduplicate using Patient ID + Admission Date when available
    if 'Patient ID' in df.columns and 'Admission Date' in df.columns:
        df = df.drop_duplicates(subset=['Patient ID','Admission Date'], keep='first')
    # drop fully-empty columns
    df = df.dropna(axis=1, how='all')
    df.to_csv(out_path, index=False)
    print(f'Wrote cleaned file: {out_path}')
    return df

In [ ]:
# Write the same cleaning function to a .py file so it can be reused outside the notebook.
pycode = '''
import pandas as pd
import numpy as np
def clean_hospital_data(in_path='hospital_cleaned.csv', out_path='hospital_cleaned_v2.csv'):
    df = pd.read_csv(in_path)
    df = df.loc[:, ~df.columns.str.contains('^Unnamed', na=False)]
    for c in df.select_dtypes(include=['object']).columns:
        df[c] = df[c].where(df[c].isna(), df[c].str.strip())
    for c in ['Hospital Name','City','State','Department','Doctor','Patient Name','Admission Type','Insurance Provider']:
        if c in df.columns:
            df[c] = df[c].astype(str).str.title().replace({'Nan': np.nan})
    for c in df.columns:
        if c.lower().endswith('id') or ' id' in c.lower() or c.lower().startswith('hospital id'):
            df[c] = df[c].astype(str).str.upper()
    for c in ['Admission Date','Discharge Date']:
        if c in df.columns:
            df[c] = pd.to_datetime(df[c], errors='coerce')
    for c in ['Admission Time','Discharge Time']:
        if c in df.columns:
            df[c] = pd.to_datetime(df[c], errors='coerce').dt.strftime('%H:%M').fillna('')
    for c in ['Age','Beds Available','ICU Beds','Staff Count','Billing Amount','Length of Stay']:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors='coerce')
    if 'Insurance Provider' in df.columns:
        df['Insurance Provider'] = df['Insurance Provider'].replace({'': np.nan}).fillna('Self-Pay')
    if 'Admission Date' in df.columns and 'Discharge Date' in df.columns:
        mask = (df['Discharge Date'].notna()) & (df['Admission Date'].notna()) & (df['Discharge Date'] < df['Admission Date'])
        df.loc[mask, 'Discharge Date'] = df.loc[mask, 'Admission Date']
    if 'Patient ID' in df.columns and 'Admission Date' in df.columns:
        df = df.drop_duplicates(subset=['Patient ID','Admission Date'], keep='first')
    df = df.dropna(axis=1, how='all')
    df.to_csv(out_path, index=False)
    print(f'Wrote cleaned file: {out_path}')
'''
with open('clean_hospital_data.py', 'w', encoding='utf-8') as f:
    f.write(pycode)
print('clean_hospital_data.py written')

In [ ]:
# Run the cleaning function on the provided CSV and show a sample
out_df = clean_hospital_data(in_path='hospital_cleaned.csv', out_path='hospital_cleaned_v2_from_notebook.csv')
out_df.head()